# **Module 6 Homework: Batch Processing with Spark**

Downloading yellow_tripdata_2025-11.parquet

In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet


--2026-03-07 22:34:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.60, 13.35.33.98, 13.35.33.10, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.3’

yellow_tripdata_202 100%[===================>]  67.84M   279MB/s    in 0.2s    

2026-03-07 22:34:50 (279 MB/s) - ‘yellow_tripdata_2025-11.parquet.3’ saved [71134255/71134255]



## **Question 1: Install Spark and PySpark**


* Install Spark
* Run PySpark
* Create a local spark session
* Execute spark.version.

In [2]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('module6_hw') \
    .getOrCreate()

In [4]:
BASE_PATH = "/content"
OUTPUT_PATH = f"{BASE_PATH}/yellow_tripdata_2025_11_parquet"

In [5]:
from pyngrok import ngrok, conf
import getpass

print("Authtoken (copy from https://dashboard.ngrok.com/get-started/your-authtoken)")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f"Ngrok tunnel is active in: {public_url}")

Authtoken (copy from https://dashboard.ngrok.com/get-started/your-authtoken)
··········


Ngrok tunnel is active in: https://aliza-prognathous-overtartly.ngrok-free.dev


What's the output?

* **A: 4.0.2**


In [6]:
print(f"Spark version: {spark.version}")

Spark version: 4.0.2


## **Question 2: Yellow November 2025**

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

In [7]:
from pyspark.sql.types import *

schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
])

df = spark.read.schema(schema).parquet('yellow_tripdata_2025-11.parquet')

In [8]:
df = df.repartition(4)
df.write.parquet(OUTPUT_PATH, mode="overwrite")

What is the average size of the Parquet (ending with .parquet extension) files that were created (in MB)? Select the answer which most closely matches.

* **A: 25MB**

In [9]:
!ls -lh {OUTPUT_PATH}

total 110M
-rw-r--r-- 1 root root 28M Mar  7 22:36 part-00000-2edf2604-a668-413b-ad17-ca9bf97f6523-c000.snappy.parquet
-rw-r--r-- 1 root root 28M Mar  7 22:36 part-00001-2edf2604-a668-413b-ad17-ca9bf97f6523-c000.snappy.parquet
-rw-r--r-- 1 root root 28M Mar  7 22:36 part-00002-2edf2604-a668-413b-ad17-ca9bf97f6523-c000.snappy.parquet
-rw-r--r-- 1 root root 28M Mar  7 22:36 part-00003-2edf2604-a668-413b-ad17-ca9bf97f6523-c000.snappy.parquet
-rw-r--r-- 1 root root   0 Mar  7 22:36 _SUCCESS


## **Question 3: Count records**

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

* **A: 162,604**

In [10]:
df.createOrReplaceTempView('yellow_trips')

In [11]:
spark.sql("""
SELECT COUNT(*)
FROM
    yellow_trips
WHERE DATE(tpep_pickup_datetime)='2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



## **Question 4: Longest trip**

What is the length of the longest trip in the dataset in hours?

* **A: 90.6**

In [12]:
spark.sql("""
SELECT
    ROUND(MAX(
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600
    ),1) AS longest_trip_hours
FROM yellow_trips
""").show()

+------------------+
|longest_trip_hours|
+------------------+
|              90.6|
+------------------+



## **Question 5: User Interface**

Spark's User Interface which shows the application's dashboard runs on which local port?

* **A: 4040**

In [13]:
spark.sparkContext.uiWebUrl

'http://4d4e3f0a61a8:4040'

## **Question 6: Least frequent pickup location zone**

Load the zone lookup data into a temp view in Spark:

In [14]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-07 22:36:38--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.98, 13.35.33.83, 13.35.33.10, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.98|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.2’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-07 22:36:38 (125 MB/s) - ‘taxi_zone_lookup.csv.2’ saved [12331/12331]



Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

* Governor's Island/Ellis Island/Liberty Island
* Arden Heights
* Rikers Island
* Jamaica Bay

If multiple answers are correct, select any

* **A: Arden Heights, Eltingville/Annadale/Prince's Bay and Governor's Island/Ellis Island/Liberty Island**

In [15]:
df_tzl = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [16]:
df_tzl.createOrReplaceTempView('taxi_zone')

In [17]:
spark.sql("""
SELECT y.PULocationID, t.Zone, count(*) as trips
FROM yellow_trips y
LEFT JOIN taxi_zone t on y.PULocationID=t.LocationID
WHERE y.tpep_pickup_datetime >='2025-11-01 00:00:00' and y.tpep_pickup_datetime<'2025-12-01 00:00:00'
GROUP BY y.PULocationID, t.Zone
ORDER BY trips ASC, t.Zone ASC
LIMIT 10
""").show(truncate=False)

+------------+---------------------------------------------+-----+
|PULocationID|Zone                                         |trips|
+------------+---------------------------------------------+-----+
|5           |Arden Heights                                |1    |
|84          |Eltingville/Annadale/Prince's Bay            |1    |
|105         |Governor's Island/Ellis Island/Liberty Island|1    |
|187         |Port Richmond                                |3    |
|109         |Great Kills                                  |4    |
|111         |Green-Wood Cemetery                          |4    |
|199         |Rikers Island                                |4    |
|204         |Rossville/Woodrow                            |4    |
|2           |Jamaica Bay                                  |5    |
|251         |Westerleigh                                  |12   |
+------------+---------------------------------------------+-----+



In [18]:
!pkill -f ngrok
ngrok.kill()

print("Ngrok Tunnel successfully closed.")

Ngrok Tunnel successfully closed.


In [19]:
spark.stop()